In [2]:
import pandas as pd

In [3]:
dataset= 'NELL995'
amie_rules_path= dataset + '/amie_mined_rules.tsv'
amie_mined_rules_aligned_path= dataset + '/amie_mined_rules_aligned.tsv'

In [4]:
amie_rules = pd.read_csv(amie_rules_path, sep='\t', header=None,
                         names=['Rule','Head Coverage','Std Confidence','PCA Confidence','Positive Examples','Body size','PCA Body size','Functional variable'])
print(len(amie_rules))
amie_rules = amie_rules[(amie_rules['Body size'] >1) & (amie_rules['Std Confidence'] <1.0)]
print(len(amie_rules))

80318
66635


In [5]:
limited_rules = amie_rules.sort_values(by= 'Std Confidence', ascending=False)[:7000]

In [6]:
def convert_rule(r):
    # Helper function to format a single triple
    def format_triple(subj, pred, obj):
        property_name = pred.split('/')[-1]
        return f"{property_name}({subj.replace('?','')},{obj.replace('?','')})"

    body_str, head_str = r.split('=>')

    head_tokens = head_str.strip().split()

    if len(head_tokens) % 3 == 0:
        for i in range(0, len(head_tokens), 3):
            hsubj, pred, hobj = head_tokens[i:i+3]
            head_formatted = format_triple(hsubj, pred, hobj)

    # Process the body
    body_tokens = body_str.strip().split()
    # if len(body_tokens) / 3  > 1:
    #     #fix the 'path' so that the head subject variable is in the first 'atom' and the head object in hte last
    #     for i in range(0, len(body_tokens), 3):
    #             chunk = body_tokens[i : i + 3]
    #             if hsubj in chunk and hobj not in chunk:
    #                 first_chunk_start_index = i
    #                 break
    #     try:
    #         first_ordered_body_tokens = body_tokens[first_chunk_start_index:] + body_tokens[:first_chunk_start_index]
    #     except Exception:
    #         print(r)
    #     first_pattern = first_ordered_body_tokens[:3]
    #     rest = first_ordered_body_tokens[3:]
    #
    #     for i in range(0, len(rest), 3):
    #             chunk = rest[i : i + 3]
    #             if hobj in chunk :
    #                 last_chunk_start_index = i
    #                 break
    #     try:
    #         last_pattern=  rest[last_chunk_start_index: last_chunk_start_index+3]
    #     except Exception:
    #         print(r)
    #     body_tokens = first_pattern + rest[:last_chunk_start_index] + rest[last_chunk_start_index+3:] + last_pattern

    body_formatted = []
    if len(body_tokens) % 3 == 0:
        for i in range(0, len(body_tokens), 3):
            subj, pred, obj = body_tokens[i:i+3]
            body_formatted.append(format_triple(subj, pred, obj))

    # Process the head



    # if hsubj not in body_tokens[:3]:
    #     print('something went wrong')
    body_result = ", ".join(body_formatted)

    return f"{head_formatted} <= {body_result}"
#
with open(amie_mined_rules_aligned_path, 'w') as outfile:
    for i,row in limited_rules.iterrows():

        rule = convert_rule(row['Rule'])
        outfile.write(str(row['Body size']) +'\t'+ str(row['Positive Examples']) + '\t' + str(row['Std Confidence']) + '\t' + rule + '\n')
# input_str = "?f  https://ste-lod-crew.fr/nell/ontology/personbelongstoorganization  ?b  ?a  https://ste-lod-crew.fr/nell/ontology/politicianusendorsedbypoliticianus  ?f  ?f  https://ste-lod-crew.fr/nell/ontology/politicianusmemberofpoliticalgroup  ?b   => ?a  https://ste-lod-crew.fr/nell/ontology/politicianusmemberofpoliticalgroup  ?b "
# convert_rule(input_str)
